# Interactive CARMENES RFI Scaling Notebook

This notebook searches the CARMENES CSV for the targets NGC5638 and SAG_DIR, selects candidate rows, and plots waterfall data at progressively wider frequency spans to reveal whether a narrowband feature sits inside a larger RFI structure.

In [ ]:
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from blimpy import Waterfall

# Project paths
ROOT = Path('/home/wlll2x/turboSETI')
CSV_PATH = ROOT / 'scripts' / 'carmenes_1688mhz_subset.csv'
OUTPUT_DIR = ROOT / 'notebooks' / 'outputs' / 'carmen_rfi_interactive'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Helper functions

def load_csv_candidates(csv_path: Path, targets: List[str]) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df[df['Target'].astype(str).str.strip().isin(targets)].copy()
    df['h5_path'] = df['.h5 path'].astype(str).str.strip()
    df['Frequency'] = pd.to_numeric(df['Frequency'], errors='coerce')
    return df.reset_index(drop=True)


def resolve_h5_path(row: pd.Series) -> Optional[Path]:
    path = row.get('h5_path', '')
    if not isinstance(path, str):
        return None
    candidate = Path(path)
    if candidate.exists():
        return candidate
    alt = Path('/datax/scratch/wlll2x/raw') / candidate.name
    if alt.exists():
        return alt
    return None


def open_waterfall(path: Path, f_start: Optional[float] = None, f_stop: Optional[float] = None):
    if f_start is None or f_stop is None:
        wf = Waterfall(str(path), load_data=True)
    else:
        wf = Waterfall(str(path), f_start=f_start, f_stop=f_stop, load_data=True)
    return wf


def get_waterfall_arrays(wf) -> Tuple[np.ndarray, np.ndarray, float]:
    data = wf.data
    if data.ndim == 3 and data.shape[1] >= 1:
        data = data[:, 0, :]
    if data.ndim != 2:
        raise ValueError(f'Unexpected waterfall shape: {data.shape}')
    freqs = wf.header.get('fch1') + np.arange(data.shape[1], dtype=float) * wf.header.get('foff', 1.0)
    tsamp = float(wf.header.get('tsamp', 1.0))
    return data, freqs, tsamp


def plot_waterfall(path: Path, center_freq: float, span_mhz: float, title: Optional[str] = None):
    f_start = center_freq - span_mhz / 2.0
    f_stop = center_freq + span_mhz / 2.0
    wf = open_waterfall(path, f_start=f_start, f_stop=f_stop)
    data, freqs, tsamp = get_waterfall_arrays(wf)
    data = np.asarray(data, dtype=float)
    data = np.clip(data, 1e-6, None)
    data_db = 10.0 * np.log10(data)
    duration_s = data.shape[0] * tsamp
    extent = (float(freqs[0]), float(freqs[-1]), 0.0, float(duration_s))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.imshow(data_db, aspect='auto', origin='lower', extent=extent, cmap='viridis')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('Time [s]')
    ax.set_title(title or f'{path.name} @ {center_freq:.4f} MHz, span {span_mhz:.3f} MHz')
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def plot_candidate(path: Path, center_freq: float, span_mhz: float):
    plot_waterfall(path, center_freq=center_freq, span_mhz=span_mhz, title=f'Waterfall around {center_freq:.4f} MHz')

print('Loaded helper functions.')

Loaded helper functions.


/home/wlll2x/.conda/envs/willvenv/lib/python3.10/site-packages/blimpy/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:
targets = ['NGC5638', 'SAG_DIR']
existing_targets = sorted({t for t in df['Target'].astype(str).str.strip() if t})
missing_targets = [t for t in targets if t not in existing_targets]

df = load_csv_candidates(CSV_PATH, targets)
print('Requested targets:', targets)
print('Found targets in CSV:', existing_targets)
if missing_targets:
    print('Missing from CSV:', missing_targets)
print(df[['Target', 'Session', 'Cadence ID', 'Frequency', '.h5 path']].head(20))
print(f'Rows matched: {len(df)}')

NameError: name 'df' is not defined

In [ ]:
candidate_rows = df[df['Frequency'].notna()].copy()
if candidate_rows.empty:
    raise RuntimeError('No candidate rows found for the selected targets.')

candidate_rows[['Target', 'Session', 'Cadence ID', 'Frequency', 'h5_path']].head(10)

In [ ]:
# Try to pick the first resolvable candidate row for a quick walkthrough.
row = candidate_rows.iloc[0]
resolved_path = resolve_h5_path(row)
print('Selected row:')
print(row[['Target', 'Session', 'Cadence ID', 'Frequency', 'h5_path']])
print('Resolved HDF5 path:', resolved_path)

In [ ]:
if resolved_path is None:
    raise FileNotFoundError(f'Could not resolve an HDF5 file for {row["Target"]}')

wf = open_waterfall(resolved_path)
print('Header keys:', sorted(wf.header.keys()))
print('fch1:', wf.header.get('fch1'))
print('foff:', wf.header.get('foff'))
print('nchans:', wf.header.get('nchans'))
print('tsamp:', wf.header.get('tsamp'))

In [ ]:
plot_candidate(resolved_path, center_freq=float(row['Frequency']), span_mhz=0.01)

In [ ]:
span_slider = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-3,  # 0.001
    max=1,   # 10.0
    step=0.1,
    description='span [MHz]',
    readout_format='.3f',
)

output = widgets.Output()

@output.capture(clear_output=True)
def update_plot(span_mhz):
    plot_candidate(resolved_path, center_freq=float(row['Frequency']), span_mhz=span_mhz)

widgets.interactive(update_plot, span_mhz=span_slider)

In [ ]:
# Show the same center frequency at wider spans to reveal broader RFI structure.
for span in [0.01, 0.1, 1.0, 10.0]:
    plot_candidate(resolved_path, center_freq=float(row['Frequency']), span_mhz=span)
    plt.close('all')

In [ ]:
# Save the initial narrow and wide-span plots for later inspection.
plot_candidate(resolved_path, center_freq=float(row['Frequency']), span_mhz=0.01)
plt.savefig(OUTPUT_DIR / f'{row["Target"]}_narrow.png', dpi=200, bbox_inches='tight')
plot_candidate(resolved_path, center_freq=float(row['Frequency']), span_mhz=1.0)
plt.savefig(OUTPUT_DIR / f'{row["Target"]}_midres.png', dpi=200, bbox_inches='tight')
print(f'Saved plots to {OUTPUT_DIR}')